# Avaliação com Qwen3-8B
Abra este notebook no Colab e selecione **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4**. A primeira célula monta o Google Drive e clona a branch `feature/llm-judge` para `/content/Amostragem-Adaptativa`, incluindo os módulos, testes e `respostas.jsonl`. A saída de cada experimento, inclusive `checkpoint.sqlite`, fica em `Meu Drive/Amostragem-Adaptativa/saidas_juiz/`. Use uma sessão por pasta de saída. O juiz Qwen3-8B é carregado somente quando um julgamento real começa, em 4 bits NF4 com cálculo float16.

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess
import sys

drive.mount('/content/drive')
raiz = Path('/content/Amostragem-Adaptativa')
branch = 'feature/llm-judge'
repo = 'https://github.com/AVTBrasil/Amostragem-Adaptativa.git'
if not raiz.exists():
    subprocess.run(['git', 'clone', '--branch', branch, '--single-branch', '--depth', '1', repo, str(raiz)], check=True)
else:
    atual = subprocess.run(['git', '-C', str(raiz), 'branch', '--show-current'], check=True, capture_output=True, text=True).stdout.strip()
    assert atual == branch, f'Branch inesperada: {atual}'
    subprocess.run(['git', '-C', str(raiz), 'pull', '--ff-only', 'origin', branch], check=True)
for arquivo in ('avaliacao/__main__.py', 'avaliacao/judge.py', 'avaliacao/requirements.txt', 'tests/test_avaliacao.py', 'respostas.jsonl'):
    assert (raiz / arquivo).is_file(), f'Arquivo ausente: {arquivo}'
pasta_drive = Path('/content/drive/MyDrive/Amostragem-Adaptativa/saidas_juiz')
pasta_drive.mkdir(parents=True, exist_ok=True)
print(f'Código: {raiz} | Resultados e checkpoints: {pasta_drive}')

## Verificação da GPU, dependências e testes com mocks

In [ ]:
import torch
assert torch.cuda.is_available(), 'Selecione GPU T4 no tipo de ambiente de execução do Colab.'
gpu = torch.cuda.get_device_name(0)
assert 'T4' in gpu, f'Este notebook foi preparado para GPU T4; GPU atual: {gpu}'
print(f'GPU: {gpu}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(raiz / 'avaliacao/requirements.txt')], check=True)

In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=raiz, check=True)

## Escolha do experimento
A opção inicial executa apenas os testes acima. Para iniciar ou retomar, selecione o benchmark e execute as células seguintes.

In [ ]:
experimento = 'apenas testes' # @param ["apenas testes", "gsm8k", "mtbench"]
saida = pasta_drive / experimento

## GSM8K

In [ ]:
if experimento == 'gsm8k':
    subprocess.run([sys.executable, '-m', 'avaliacao', 'gsm8k', '--entrada', str(raiz / 'respostas.jsonl'), '--saida', str(saida)], cwd=raiz, check=True)

## MT-Bench

In [ ]:
if experimento == 'mtbench':
    subprocess.run([sys.executable, '-m', 'avaliacao', 'mtbench', '--saida', str(saida)], cwd=raiz, check=True)

## Tabelas e gráficos

In [ ]:
if experimento in ('gsm8k', 'mtbench'):
    import pandas as pd
    from IPython.display import display, Image
    for arquivo in sorted(saida.glob('*.csv')):
        print(arquivo.name)
        display(pd.read_csv(arquivo))
    for arquivo in sorted(saida.glob('*.png')):
        display(Image(filename=str(arquivo)))